# LoRA SFT v4 — Full Corpus + Gravity Format Fix

**Goal:** v3 (LB 0.58) confirmed data scaling works. v4 adds one targeted fix: the gravity solver now preserves gold's decimal precision (drops trailing zero when applicable). This unlocks 114 previously-filtered gravity training records that have 1-decimal gold (e.g. `57.0` not `57.00`), and teaches the model to match gold's format convention for 9.8% of gravity problems.

**Data:** 4795 records from `gideonnazarian/nemotron-solver-training-data` (v4) — same as v3 (4681) plus 114 new gravity examples with 1-decimal answers. Same Fix A reasoning structure for all categories.

Category counts: numeral_system 1576, unit_conversion 1328, **gravity 1136 (+114 vs v3)**, cipher 605, bit_manipulation 150.

**Training:** identical hyperparams to v3 (lr 2e-4, 1 epoch, effective batch 8, LoRA r=32, MAX_SEQ_LEN=1024). Only variable changed from v3→v4 is the gravity solver's output format. Expected step count: ~600 (~+15 steps from 114 new examples). Expected walltime: ~5-6 hours.

**Reference points:**
- Untrained baseline LB: 0.500
- v1 (template overfit): 0.510
- v2 (Fix A, 750 records): 0.540
- v3 (Fix A, full 4681): **0.580**
- Tong Huikang's progress prize: 0.850

**Expected lift:** +0.01-0.03 LB. 9.8% of gravity test problems likely have 1-decimal gold; the model currently never sees this format in training.

In [ ]:
# Cell 1: Setup — sitedir discovery + ptxas-blackwell workaround + seeds
import os, random, shutil, site, stat, sys, time
import numpy as np
import torch

T0 = time.perf_counter()
def step(label):
    print(f"[{time.perf_counter() - T0:7.1f}s] {label}", flush=True)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Find python_packages from attached utility scripts
for utility_root in ("/kaggle/input/notebooks", "/kaggle/usr/lib/notebooks", "/kaggle/lib"):
    if not os.path.isdir(utility_root):
        continue
    for root, dirs, _ in os.walk(utility_root):
        if "models" in root.split(os.sep):
            dirs[:] = []
            continue
        if os.path.basename(root) == "python_packages":
            site.addsitedir(root)

# ptxas-blackwell exec workaround (Triton needs an executable copy)
PTXAS_DST = "/tmp/ptxas-blackwell"
for src in (
    "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/triton/backends/nvidia/bin/ptxas-blackwell",
    "/kaggle/input/notebooks/ryanholbrook/nvidia-utility-script/triton/backends/nvidia/bin/ptxas-blackwell",
):
    if os.path.exists(src):
        if not os.path.exists(PTXAS_DST):
            shutil.copy(src, PTXAS_DST)
            os.chmod(PTXAS_DST, stat.S_IRWXU | stat.S_IRGRP | stat.S_IXGRP | stat.S_IROTH | stat.S_IXOTH)
        os.environ["TRITON_PTXAS_PATH"] = PTXAS_DST
        os.environ["TRITON_PTXAS_BLACKWELL_PATH"] = PTXAS_DST
        break

step(f"Setup done; seeds set to {SEED}")

In [ ]:
# Cell 2: Load model + tokenizer + apply LoRA
import kagglehub
import mamba_ssm  # noqa: F401
from peft import LoraConfig, TaskType, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
step("Tokenizer loaded")

# Tokens used for the response boundary
IM_END_ID = tokenizer.convert_tokens_to_ids("<|im_end|>")
step(f"IM_END_ID={IM_END_ID}; eos_token_id={tokenizer.eos_token_id}")

t = time.perf_counter()
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, device_map="auto", trust_remote_code=True, dtype=torch.bfloat16,
)
step(f"Model loaded in {time.perf_counter() - t:.0f}s, VRAM={torch.cuda.memory_allocated() / 1e9:.1f} GB")

lora_cfg = LoraConfig(
    r=32, lora_alpha=16,
    target_modules=r".*\.(in_proj|out_proj|up_proj|down_proj)$",
    lora_dropout=0.05, bias="none", task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

# Enable gradient checkpointing to save memory during backward
model.gradient_checkpointing_enable()
model.enable_input_require_grads()
step("LoRA wrapped + gradient checkpointing on")

In [ ]:
# Cell 3: Load training data + tokenize
import json
import glob

PROMPT_SUFFIX = (
    "\nPlease put your final answer inside `\\boxed{}`. "
    "For example: `\\boxed{your answer}`"
)

# v3 uses the FULL 4681-record corpus (training-data.jsonl) rather than the
# 750-record stratified subset (training-stratified.jsonl) used by v1/v2.
# Kaggle datasets mount under /kaggle/input/datasets/<user>/<slug>/.
CANDIDATE_PATHS = [
    "/kaggle/input/datasets/gideonnazarian/nemotron-solver-training-data/training-data.jsonl",
    "/kaggle/input/nemotron-solver-training-data/training-data.jsonl",
]
DATA_PATH = None
for p in CANDIDATE_PATHS:
    if os.path.exists(p):
        DATA_PATH = p
        break
if DATA_PATH is None:
    hits = glob.glob("/kaggle/input/**/training-data.jsonl", recursive=True)
    if hits:
        DATA_PATH = hits[0]
if DATA_PATH is None:
    raise FileNotFoundError(
        f"training-data.jsonl not found. /kaggle/input/ contains: {os.listdir('/kaggle/input')}"
    )
step(f"Using DATA_PATH={DATA_PATH}")

records = []
with open(DATA_PATH) as f:
    for line in f:
        records.append(json.loads(line))
step(f"Loaded {len(records)} training records")

MAX_SEQ_LEN = 1024

def tokenize_example(record):
    user_text = record["prompt"] + PROMPT_SUFFIX
    # Get the prompt as TEXT first (avoids BatchEncoding return-type quirks),
    # then encode to plain list[int].
    prompt_text = tokenizer.apply_chat_template(
        [{"role": "user", "content": user_text}],
        tokenize=False, add_generation_prompt=True,
    )
    prompt_ids = tokenizer.encode(prompt_text, add_special_tokens=False)
    response_text = (
        record["reasoning"] + "\n</think>\n\\boxed{" + record["answer"] + "}<|im_end|>"
    )
    response_ids = tokenizer.encode(response_text, add_special_tokens=False)
    full_ids = prompt_ids + response_ids
    if len(full_ids) > MAX_SEQ_LEN:
        return None
    labels = [-100] * len(prompt_ids) + response_ids
    return {
        "input_ids": full_ids,
        "labels": labels,
        "attention_mask": [1] * len(full_ids),
    }

examples = []
skipped = 0
for rec in records:
    ex = tokenize_example(rec)
    if ex is None:
        skipped += 1
    else:
        examples.append(ex)
step(f"Tokenized {len(examples)} examples ({skipped} skipped > {MAX_SEQ_LEN} tokens)")

lens = [len(ex["input_ids"]) for ex in examples]
step(f"Token-length stats: min={min(lens)}, median={sorted(lens)[len(lens)//2]}, max={max(lens)}")

# Sanity-print first example's decoded prompt+response so we can verify format
ex0 = examples[0]
print("\n=== Example 0 sanity check ===")
prompt_part = tokenizer.decode([t for t, l in zip(ex0["input_ids"], ex0["labels"]) if l == -100])
response_part = tokenizer.decode([l for l in ex0["labels"] if l != -100])
print(f"PROMPT (loss masked, last 200 chars): ...{prompt_part[-200:]!r}")
print(f"RESPONSE (training target): {response_part!r}")

In [ ]:
# Cell 4: Train with HF Trainer
from torch.utils.data import Dataset
from transformers import Trainer, TrainingArguments

class SFTDataset(Dataset):
    def __init__(self, examples):
        self.examples = examples
    def __len__(self):
        return len(self.examples)
    def __getitem__(self, idx):
        return self.examples[idx]

def collate(batch):
    # Right-pad to max length in batch
    max_len = max(len(ex["input_ids"]) for ex in batch)
    input_ids, labels, attn = [], [], []
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
    for ex in batch:
        n_pad = max_len - len(ex["input_ids"])
        input_ids.append(ex["input_ids"] + [pad_id] * n_pad)
        labels.append(ex["labels"] + [-100] * n_pad)
        attn.append(ex["attention_mask"] + [0] * n_pad)
    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long),
        "attention_mask": torch.tensor(attn, dtype=torch.long),
    }

train_ds = SFTDataset(examples)

args = TrainingArguments(
    output_dir="/kaggle/working/lora_output",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=1,
    learning_rate=2e-4,
    lr_scheduler_type="linear",
    warmup_ratio=0.05,
    logging_steps=5,
    save_strategy="no",
    bf16=True,
    optim="adamw_torch",
    remove_unused_columns=False,
    report_to="none",
    seed=SEED,
    gradient_checkpointing=False,  # already enabled on model above
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    data_collator=collate,
)

step("Starting training")
trainer.train()
step("Training done")

In [ ]:
# Cell 5: Save adapter to /kaggle/working and zip submission
OUTPUT_DIR = "/kaggle/working"

# Clean up any stale adapter files from prior runs
for f in os.listdir(OUTPUT_DIR):
    if f.startswith("adapter") or f.endswith(".safetensors") or f.endswith(".bin"):
        try:
            os.remove(os.path.join(OUTPUT_DIR, f))
        except (IsADirectoryError, PermissionError):
            pass

# Save trained adapter
model.save_pretrained(OUTPUT_DIR)
step("Adapter saved")

# Move any nested files up to /kaggle/working level (Trainer may have nested under lora_output/)
import subprocess
subprocess.run("find /kaggle/working -name 'adapter*' -exec mv {} /kaggle/working/ \\;", shell=True, check=False)
subprocess.run("ls -la /kaggle/working/", shell=True, check=False)

# Zip everything for submission
subprocess.run("cd /kaggle/working && rm -rf lora_output && zip -m submission.zip adapter*", shell=True, check=True)
step("submission.zip created")

In [ ]:
step(f"Total wall time: {(time.perf_counter() - T0) / 60:.1f} min")
print("Done — ready for Submit to Competition.")